# KKBox — Churn Prediction Results

**Purpose:** show how well simple rules and two models rank the users who will not renew, at a limited intervention capacity (the top 5%, 10% and 20% of each month's expiring users), with time-ordered validation.

**Inputs:** the saved tables in `reports/model_results/` written by `python -m src.run_experiments` (about 25 minutes) and `python -m src.run_ablation` (about 10 minutes). This notebook only reads them, so it runs in seconds. The protocol, leakage controls and limits are in [docs/prediction_design.md](../docs/prediction_design.md), [reports/model_results.md](../reports/model_results.md) and the [model card](../docs/model_card.md).

**Evidence labels:** full-data results on every user of each evaluated month; settings were fixed (no tuning), so the test months were never used to choose anything. Associations, not causes.

**Reading order:** setup → validation months → final test → external March check → feature families → by lead time → calibration.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

sys.path.insert(0, str(Path.cwd().parent))
from src.audit_display import show

RESULTS = Path.cwd().parent / "reports" / "model_results"
results = pd.read_csv(RESULTS / "results.csv")
show("Saved results", results.groupby(["split", "month"]).size().rename("rows").to_frame(),
     note="One row per model and capacity; 21 rows per month = 7 models or rules x 3 capacities.")

### 1. Validation months (2016-07 to 2016-12, mean of six months)

**What to look for:** the models against the rules. Recall at K is the share of all churners reached by contacting the top K% of users; a random list reaches K%. Lift is the precision of the list divided by the churn rate of the month (random = 1).

In [ ]:
validation = results[results.split == "validation"]
table = validation.groupby(["model", "capacity"])[["recall", "lift"]].mean().unstack("capacity")
table.columns = [f"{name}_{int(100 * k)}pct" for name, k in table.columns]
table["auc"] = validation.groupby("model")["auc"].mean()
show("Validation: mean over six months", table.sort_values("recall_10pct", ascending=False),
     note="Recall and lift at 5, 10 and 20 percent capacity; AUC is the area under the ROC curve over all users.")

### 2. Final test (2017-01 and 2017-02, rebuilt labels)

**What to look for:** gradient boosting and logistic regression against the best rule ("no auto-renew"), and the base churn rate of each month.

In [ ]:
test = results[results.split == "test"]
show("Final test: recall at each capacity", test.pivot_table(index="model", columns=["month", "capacity"], values="recall"))
show("Final test: lift at each capacity", test.pivot_table(index="model", columns=["month", "capacity"], values="lift"))
show("Final test: AUC and churn rate", test.groupby(["month", "model"]).agg(auc=("auc", "first"), brier=("brier", "first"),
                                                                           churn_rate=("base_rate", "first")))

### 3. External check (2017-03, official labels)

**What to look for:** the official labels treat early renewers differently and include users outside the rebuilt population, so this month is reported separately. Does the model still beat the auto-renew rule at each capacity?

In [ ]:
external = results[results.split == "external"]
show("External check: recall at each capacity", external.pivot_table(index="model", columns="capacity", values="recall"))
show("External check: AUC and churn rate", external.groupby("model").agg(auc=("auc", "first"), churn_rate=("base_rate", "first")))
display(Image(filename=str(Path.cwd().parent / "reports" / "figures" / "recall_at_capacity.png")))

### 4. Which feature families matter (gradient boosting)

**What to look for:** how much the listening features add on top of the subscription record, and how much they achieve alone.

In [ ]:
ablation = pd.read_csv(RESULTS / "ablation.csv")
show("Feature families: recall at 10 percent capacity", ablation[ablation.capacity == 0.10].pivot_table(index="features", columns="month", values="recall"))
show("Feature families: AUC", ablation.groupby(["features", "month"])["auc"].first().unstack("month"))
importance = pd.read_csv(RESULTS / "feature_importance_2017-01.csv")
show("Permutation importance (AUC drop, January 2017, top 10)", importance.head(10).set_index("feature"))

### 5. By lead time and calibration

**What to look for:** users who expire soon are easier to find but leave less time to act; and whether predicted probabilities match observed churn rates (the last decile in each month).

In [ ]:
bands = pd.read_csv(RESULTS / "lead_time_bands.csv")
bands = bands[bands.model.isin(["gradient boosting", "rule: no auto-renew"]) & bands.band.ne("nan")]
show("Top 10 percent list: churners reached by lead time (days to expiration)",
     bands.pivot_table(index=["month", "model"], columns="band", values="recall")[["1-7", "8-14", "15-21", "22-31"]])
calibration = pd.read_csv(RESULTS / "calibration.csv")
show("Calibration, gradient boosting: predicted against observed churn rate by risk decile (10 = highest)",
     calibration[calibration.model == "gradient boosting"].pivot_table(index="bin", columns="month", values=["mean_predicted", "observed_rate"]))

### 6. Findings

- **On the rebuilt labels the models clearly beat the baselines.** Contacting 10% of the expiring users with the gradient boosting list reaches 71.9% (January) and 68.1% (February 2017) of the churners, against 52.5% and 54.2% for the "no auto-renew" rule and 10% for a random list; lift at 5% capacity is about 9.8 and 9.7. Logistic regression is within 1 point of gradient boosting.
- **On the official March labels the advantage over the auto-renew rule disappears at 10% capacity** (44.9% against 45.9%); the model is better at 5% (27.9% against 22.9%) and 20% (65.6% against 58.4%). Part of the gap is population: 108,802 official users fall outside the rebuilt population rule and churn at 22.7%; the rest is the open question about early renewers.
- **Listening behavior adds little on top of the subscription record:** removing the listening features changes recall at 10% by about 1 point in January and February and improves March; listening alone reaches 30.0% and 27.6% at 10% (about three times random). The profile columns add no consistent gain.
- **The largest single driver is the last payment method code** (permutation AUC drop 0.093), whose meaning is unknown. The next ones are days since the last transaction, number of transactions, auto-renew, and a cancellation as the last row.
- **Calibration** is good on January and February and under-predicts in March. Users who expire within 7 days are the easiest to find (88.0% and 81.9% of their churners reached at 10%).
- **Limits:** three evaluation months with falling AUC (0.885, 0.865, 0.810), rebuilt labels differ from the official ones, predicted risk is not the likelihood of responding to an offer, and no effect of any action is claimed.